# Lesson 12 - エージェントスクラッチパッドによるチャット履歴の削減

このノートブックでは、Microsoft Agent Frameworkを使用して長い会話のコンテキストを管理する方法を示します。会話が長くなるとトークン数が増加し、最終的にモデルのコンテキストウィンドウを超えてしまいます。これに対処するために、**コンテキスト要約パターン**と、永続的なメモリを持つ**エージェントスクラッチパッド**を使用します。

## 学習内容:
1. **なぜコンテキスト管理が重要か**：トークン制限とコンテキストウィンドウの理解
2. **コンテキスト認識エージェント**：自身の会話コンテキストを管理するエージェントの構築
3. **コンテキスト要約パターン**：会話履歴を圧縮するためのツールの使用
4. **エージェントスクラッチパッド**：コンテキスト縮小を乗り越える永続メモリ

## 前提条件:
- 環境変数が設定されたAzure OpenAIのセットアップ
- 前のレッスンで学んだ基本的なエージェントの概念の理解


## セットアップ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## コンテキスト管理が重要な理由

すべてのLLMには有限の**コンテキストウィンドウ**があります — 一度のリクエストで処理できる最大トークン数です。会話が複数ターンに進むにつれて：

- **トークン数は各ユーザーメッセージとアシスタントの返信ごとに線形に増加**します。
- **プロンプトトークンがコストの大部分を占める**のは、会話履歴全体が毎ターン再送信されるためです。
- 最終的に会話が**コンテキストウィンドウを超え**、モデルは切り捨てるかエラーを起こします。

### コンテキスト管理の戦略

| 戦略 | 仕組み | トレードオフ |
|---|---|---|
| **切り捨て** | 古いメッセージを削除 | 初期のコンテキストを失う |
| **要約** | 古いメッセージをまとめて要約 | 一部の詳細は失うが重要なポイントは保持 |
| **スクラッチパッド / 外部メモリ** | 会話外部に重要な事実を保存 | ツール呼び出しが必要だが、どんな削減にも耐えられる |

このノートブックでは、会話履歴が凝縮されてもエージェントが連続性を維持できるように、**要約**と**スクラッチパッドツール**を組み合わせています。


## コンテキスト認識エージェントの作成


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## 長い会話のシミュレーション

コンテキストがどのように積み重なるかを見るために、複数ターンの会話を進めてみましょう。エージェントはターンを通じて主要な詳細（好み、予算、旅行日程）を保持し、連続性を示す必要があります。


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

エージェントが以前のやり取りから文脈を保持していることに注目してください — 日本、寿司、寺院、写真撮影、3000ドルの予算、一人旅、そして4月の期間について知っています。短い会話ではこれがうまく機能しますが、会話が長くなると過去の全履歴を送り直すコストが高くなります。

文脈の蓄積を確認するために、さらにやり取りを続けてみましょう：


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## コンテキスト要約パターン

会話が進むにつれて、蓄積されたコンテキストをコンパクトな形式に凝縮するために**要約ツール**を使用できます。エージェントはこのツールを呼び出して主要な好みを記録し、古いメッセージが削除されても重要な情報が保持されるようにします。

このパターンは、より高度な履歴削減の基礎となります：
1. エージェントが会話から重要な事実を特定する
2. 要約ツールを呼び出してそれらを保存する
3. 要約が重要な点を捉えているため、古いメッセージは安全に削除できる

以下に、エージェントがこれまでに学んだことのコンパクトな要約を記録するために呼び出せる `summarize_preferences` ツールを定義します。


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## 概要

このレッスンでは、Microsoft Agent Framework を使用して、長時間にわたるエージェントの会話でコンテキストを管理する方法を学びました。

### 重要な概念
- **コンテキストウィンドウは有限である** — 会話履歴のすべてのトークンはコストがかかり、制限にカウントされます。
- **要約ツール** は、蓄積されたコンテキストをコンパクトな要約に凝縮し、トークン使用量を削減しつつ重要な情報を保持します。
- **エージェントのスクラッチパッド** は、会話の縮小後も持続する外部メモリを提供します。

### 作成したもの
- 複数ターンの会話を通じて継続性を維持する **コンテキスト対応エージェント**
- ユーザーの重要な詳細をコンパクトな形式で記録する **要約ツール** (`summarize_preferences`)
- コンテキストの保持と変更処理を示す **複数ターンの会話**

### 実世界での応用
- **カスタマーサービスボット**: 長時間のサポートセッションでの好みを記憶
- **パーソナルアシスタント**: コンテキストを再説明せずに進行中のプロジェクトを追跡
- **教育チューター**: 多数の対話を通じて生徒の進捗を維持

### 次のステップ
- ファイルベースの永続化を備えた完全なスクラッチパッドツールを実装
- 要約後の自動履歴切り詰めを追加
- セマンティックメモリ検索のためにベクターデータベースと組み合わせ
- 数日後に完全なコンテキストで会話を再開できるエージェントを構築


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：  
本書類はAI翻訳サービス「Co-op Translator」（https://github.com/Azure/co-op-translator）を使用して翻訳されました。正確性には努めておりますが、自動翻訳には誤りや不正確な部分が含まれる可能性があります。原文の母国語版が権威ある参照資料とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じた誤解や誤訳について、当方は一切の責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
